In [1]:
import numpy as np
import math

def getDirSignArray(nAtoms, centralCaAtomIdx):
    """
    중심 residue의 CA 원자를 기준으로 방향성 부호 배열을 생성.

    이는 거리 기반 score 계산 시 방향 정보를 반영하는 데 사용됨.

    Args:
        nAtoms (int): 전체 원자 수 (보통 단백질 전체 residue 수)
        centralCaAtomIdx (int): 중심 residue의 CA 원자 인덱스

    Returns:
        dirSignArray (np.ndarray): (nAtoms,) 크기의 배열로,
            중심 이전 residue는 -1, 이후 residue는 +1로 설정됨.
    """
    indexRange = np.arange(0, nAtoms, dtype=np.int32)
    dirSignArray = np.where(indexRange < centralCaAtomIdx, -1, 1).astype(np.int32)
    return dirSignArray

def dot_numba(A,B):
    m, n = A.shape
    p = B.shape[1]

    C = np.zeros((m,p), dtype=np.float32)

    for i in range(0,m):
        for j in range(0,p):
            for k in range(0,n):
                C[i,j] += A[i,k]*B[k,j] 
    return C

def centerAndRandomRotate_helper(atomCoords, centerCoord, rotMatrix, rotate):
    """
    원자 좌표(atomCoords)를 중심(centerCoord) 기준으로 정렬하고,
    선택적으로 주어진 회전행렬(rotMatrix)을 적용하여 랜덤 회전시킴.

    Args:
        atomCoords: (N, 3) 형태의 원자 좌표
        centerCoord: 중심으로 삼을 좌표 (예: 중심 residue의 CA 위치)
        rotMatrix: (3, 3) 회전행렬
        rotate: 회전 적용 여부 (True/False)

    Returns:
        중심 정렬 및 회전된 좌표 (atomCoords_ret)
    """

    atomCoords_ret = np.empty(atomCoords.shape, dtype=np.float32)
    for i in range(atomCoords.shape[0]):
        atomCoords_ret[i, 0] = atomCoords[i, 0] - centerCoord[0]
        atomCoords_ret[i, 1] = atomCoords[i, 1] - centerCoord[1]
        atomCoords_ret[i, 2] = atomCoords[i, 2] - centerCoord[2]
    

    if rotate:
        atomCoords_ret = dot_numba( atomCoords_ret, rotMatrix )

    return atomCoords_ret

def centerAndRandomRotate(atomCoords, centerCoord, rotMats, rotate):
    """
    주어진 회전행렬 중 하나를 랜덤 선택하거나, 회전 없이 단위행렬을 사용하여
    중심정렬 및 랜덤 회전을 수행하는 상위 함수.

    Args:
        atomCoords: (N, 3) 원자 좌표
        centerCoord: 중심 좌표
        rotMats: 사전 정의된 (R, 3, 3) 회전 행렬 리스트
        rotate: 회전 여부

    Returns:
        정렬 및 회전된 좌표
    """

    if rotate:
        rotMat = rotMats[np.random.randint(rotMats.shape[0]),:]
    else:
        rotMat = np.array( [[1.,  0.,  0.], \
                            [0.,  1.,  0.], \
                            [0.,  0.,  1.]], dtype=np.float32)

    atomCoords_ret = centerAndRandomRotate_helper(atomCoords, centerCoord, rotMat, rotate)
    
    return atomCoords_ret

def getTargetBoxIdxs(allCoords, resIds, change_position_1based, edgeLen, excludeCentralAA):
    """
    모든 원자 좌표 중 voxel box 범위(edgeLen) 안에 위치한 원자만 필터링.
    중심 변이 residue를 제외할 수도 있음.

    Args:
        allCoords (np.ndarray): (N, 3) 형태의 원자 좌표들
        resIds (np.ndarray): 각 원자의 residue ID (1-based indexing)
        change_position_1based (int): 중심 변이가 발생한 residue ID
        edgeLen (float): 박스의 절반 길이. 즉, -edgeLen ~ +edgeLen 내의 박스.
        excludeCentralAA (bool): 중심 residue를 제외할지 여부

    Returns:
        boxIdx (np.ndarray): (N,) 크기의 bool 배열. box 내부에 속하고,
            필요시 중심 residue가 아닌 경우 True.
    """
    boxIdx = np.empty( allCoords.shape[0], dtype=np.bool_ )
    
    if excludeCentralAA:
        for i in range(allCoords.shape[0]):
            boxIdx[i] = (allCoords[i][0] < edgeLen and allCoords[i][0] > -edgeLen) and \
                        (allCoords[i][1] < edgeLen and allCoords[i][1] > -edgeLen) and \
                        (allCoords[i][2] < edgeLen and allCoords[i][2] > -edgeLen) and \
                        resIds[i] != change_position_1based
    else:
        for i in range(allCoords.shape[0]):
            boxIdx[i] = (allCoords[i][0] < edgeLen and allCoords[i][0] > -edgeLen) and \
                        (allCoords[i][1] < edgeLen and allCoords[i][1] > -edgeLen) and \
                        (allCoords[i][2] < edgeLen and allCoords[i][2] > -edgeLen)
    
    return boxIdx

def getGridCenters(nVoxels, center, voxelSize):
    """
    중심 중심좌표(center)를 기준으로 3D voxel grid의 각 셀의 중심 좌표 계산.

    Args:
        nVoxels: [x, y, z] 방향의 voxel 개수 (홀수여야 함)
        center: 전체 voxel box의 중심 좌표
        voxelSize: voxel 한 변의 길이 (Å 단위)

    Returns:
        (num_voxels, 3) 형태의 각 voxel 중심 좌표 리스트
    """

    nVoxels = np.array(nVoxels)
    
    if nVoxels[0] % 2 == 0:
        raise Exception("Number of voxels must be odd!")
    
    x, y, z = nVoxels

    firstdim = np.repeat(np.arange(x) * voxelSize, y*z)

    seconddim = np.tile(np.repeat(np.arange(y) * voxelSize, z), x)

    thirddim = np.tile(np.arange(z) * voxelSize, x*y)

    combined = np.vstack((firstdim.T, seconddim.T, thirddim.T)).T.astype(np.float64)
    combined = combined.reshape([x, y, z, 3])

    nVoxelsSide = (nVoxels[0] - 1) / 2 
    minCenter = center - (nVoxelsSide * voxelSize)
    
    centers = combined + minCenter

    centers = centers.reshape(np.prod(nVoxels), 3).copy()
    
    return centers

def withinBounds(coord, boxLen_half):
    
    if coord[0] > boxLen_half or coord[0] < -boxLen_half:
        return False
    elif coord[1] > boxLen_half or coord[1] < -boxLen_half:
        return False
    elif coord[2] > boxLen_half or coord[2] < -boxLen_half:
        return False
    
    return True

def getCenterNeighborCoords(singleCenterCoords, voxelSize_local, boxLen_half):
    
    """
    주어진 voxel 중심 주변의 26개 이웃 voxel 중심 좌표 계산 (총 3x3x3 - 1개)

    Args:
        singleCenterCoords: 기준 voxel의 중심 좌표
        voxelSize_local: voxel 크기
        boxLen_half: 전체 voxel box의 절반 길이

    Returns:
        (26, 3) 배열, 각 이웃 voxel의 중심 좌표. (범위 밖이면 값은 1000)
    """

    centerNeighborCoords = np.full((3*3*3-1, 3), 1000, dtype=np.float32)

    centerNeighborCoords_loop=np.empty(3, dtype=np.float32)
    counter = 0
    for dim_j in [-1,0,1]:
        for dim_k in [-1,0,1]:
            for dim_l in [-1,0,1]:
                if not (dim_j == 0 and dim_k == 0 and dim_l == 0):

                    centerNeighborCoords_loop[0] = singleCenterCoords[0] + dim_j*voxelSize_local
                    centerNeighborCoords_loop[1] = singleCenterCoords[1] + dim_k*voxelSize_local
                    centerNeighborCoords_loop[2] = singleCenterCoords[2] + dim_l*voxelSize_local

                    isWithinBounds = withinBounds(centerNeighborCoords_loop, boxLen_half)

                    if isWithinBounds:
                        centerNeighborCoords[counter] = centerNeighborCoords_loop

                        counter += 1
                        
    return centerNeighborCoords

def getBoxDimIdx_helper(coord, boxLen_half, voxelSize_local):
    
    idxs = np.empty(3, dtype=np.int32)
    
    coord_std = coord[0] + boxLen_half
    idxs[0] = int(coord_std / voxelSize_local)
    
    coord_std = coord[1] + boxLen_half
    idxs[1] = int(coord_std / voxelSize_local)

    coord_std = coord[2] + boxLen_half
    idxs[2] = int(coord_std / voxelSize_local)
    
    return idxs

def getCenterIdx(idxs, nVoxels_local):
    return  (idxs[2]) + \
            (idxs[1] * nVoxels_local) + \
            (idxs[0] * nVoxels_local * nVoxels_local)

def getBoxDimIdx(coords_local, boxLen_half, voxelSize_local, nVoxels_local):
    
    centerIdxs = np.empty(coords_local.shape[0], dtype=np.int32)
    
    for i in range(coords_local.shape[0]):
        idxs_i = getBoxDimIdx_helper(coords_local[i], boxLen_half, voxelSize_local)        
        centerIdxs[i] = getCenterIdx(idxs_i, nVoxels_local)
    
    return centerIdxs


def getCenterNeighborIdxs(neighborCoordArray, boxLen_half, voxelSize_local, nVoxels_local):

    """
    이웃 중심 좌표들에 대해 해당하는 voxel index(1D index) 계산

    Args:
        neighborCoordArray: (26, 3) 이웃 중심 좌표
        boxLen_half: 박스 절반 길이
        voxelSize_local: voxel 크기
        nVoxels_local: 각 축의 voxel 개수

    Returns:
        (26,) 배열, 각 이웃의 인덱스 (범위 밖이면 -1)
    """
    
    neighborIdxs = np.full(neighborCoordArray.shape[0], -1, dtype=np.int32)
    
    for i in range(neighborCoordArray.shape[0]):
        
        currNeighborCoord = neighborCoordArray[i,:]
        
        if currNeighborCoord[0] == 1000:
            break
        
        currNeighborCoord_ext = np.ascontiguousarray(currNeighborCoord).reshape((1, -1))
        
        centerIdx = getBoxDimIdx(currNeighborCoord_ext, boxLen_half, voxelSize_local, nVoxels_local)
        neighborIdxs[i] = centerIdx[0]

    return neighborIdxs

def checkNeighbors(neighborCoordArray, neighborIdxs, centerCoords):
    for i in range(neighborCoordArray.shape[0]):
        if neighborCoordArray[i,0] == 1000:
            break
        
        if not (np.all(neighborCoordArray[i,:] == centerCoords[neighborIdxs[i],:])):
            return False
    
    return True

def getCenterNeighbors(centers, voxelSize_local,  boxLen_half, nVoxels_local):

    """
    전체 voxel 중심들에 대해 각 중심별 이웃 voxel 좌표 및 인덱스 계산

    Args:
        centers: (N, 3) 전체 voxel 중심 좌표
        voxelSize_local: voxel 크기
        boxLen_half: 박스 절반 길이
        nVoxels_local: 각 축 voxel 수

    Returns:
        centerIdxToNeighborCoords: (N, 26, 3) 각 중심에 대한 이웃 중심 좌표
        centerIdxToNeighborIdxs: (N, 26) 각 중심에 대한 이웃 인덱스
        np.all(checks): 이웃 검증 결과
    """
    
    checks = []

    centerIdxToNeighborCoords = np.empty((centers.shape[0], 3*3*3-1, 3),  dtype=np.float32)
    centerIdxToNeighborIdxs = np.empty((centers.shape[0], 3*3*3-1),  dtype=np.int32)
    
    for center_i in range(centers.shape[0]):
        singleCenterCoords = centers[center_i, :]
        neighborCoordArray = getCenterNeighborCoords(singleCenterCoords, voxelSize_local, boxLen_half)
        neighborIdxs = getCenterNeighborIdxs(neighborCoordArray, boxLen_half, voxelSize_local, nVoxels_local)

        
        assert(neighborCoordArray.shape[0] == neighborIdxs.shape[0])
        
        centerIdxToNeighborCoords[center_i, :] = neighborCoordArray
        centerIdxToNeighborIdxs[center_i, :] = neighborIdxs
        
        check = checkNeighbors(centerIdxToNeighborCoords[center_i, :], centerIdxToNeighborIdxs[center_i, :], centers)
        checks.append(check)

    return centerIdxToNeighborCoords, centerIdxToNeighborIdxs, np.all(np.array(checks))

def addTriple(counter, returnIdxs, returnValues, centerIdx, featIdx, value):

    """
    단일 (center, feature, value) triple을 sparse 형태로 기록

    Args:
        counter: 현재 triple 위치
        returnIdxs: triple 인덱스 배열 (Nx2)
        returnValues: triple 값 배열 (N,)
        centerIdx: 중심 voxel 인덱스
        featIdx: 피처 인덱스
        value: 값

    Returns:
        triple이 추가되었으면 1, 아니면 0
    """

    added = 0
    if value != 0.0:
        returnIdxs[counter, 0] = centerIdx
        returnIdxs[counter, 1] = featIdx
        returnValues[counter] = value
        
        added = 1

    return added

def addTriple_batch(counter, returnIdxs, returnValues, centerIdx, featIdx, values):
    """
    여러 feature에 대한 triple을 한번에 기록하는 batch 버전

    Returns:
        추가된 triple 수
    """

    nrAdded = 0
    for i in range(values.shape[0]):
        value = values[i]
        nrAdded += addTriple(counter+nrAdded, returnIdxs, returnValues, centerIdx, featIdx+i, value)

    return nrAdded

def distToScore(dist, pointIdx, dirSignBox, maxDist):
    score = max(0, 1.0 - math.sqrt(dist / maxDist))

    if dirSignBox[pointIdx] != 1:
        score = -score
        
    return score

def getOccTuples_jigsaw( centerIdxToNnPointIdx, 
            centerIdxToNnDist, 
            centerIdxToNnDistPerAA, 
            centerIdxToNnPointIdxPerAA,
            dirSignBox, 
            protEvoArray, 
            protQualArray, 
            boxResIds, 
            nfeatsEvo, 
            nfeatsQual, 
            nfeatsSeq,
            nTargetAtoms,
            maxDist, 
            nFeatsAltRef):
            #caCbMap):

    """
    중심 voxel에 대해 여러 종류의 feature (거리 기반 score, 진화 정보, 품질, co-evolution 등)를
    triple (index, value) 형태로 변환.

    Returns:
        returnIdxs: (N, 2), 각 triple의 (voxelIdx, featureIdx)
        returnValues: (N,), 각 triple의 값
    """

    returnIdxs = np.full((centerIdxToNnPointIdx.shape[0] * (1 + nfeatsEvo + nfeatsQual + nfeatsSeq*nTargetAtoms), 2), -1, dtype=np.uint16)
    returnValues = np.full((centerIdxToNnPointIdx.shape[0] * (1 + nfeatsEvo + nfeatsQual + nfeatsSeq*nTargetAtoms)), -1, dtype=np.float32)

    counter = 0

    for i in range(centerIdxToNnPointIdx.shape[0]):
        
        centerIdx = centerIdxToNnPointIdx[i, 0]
        pointIdx = centerIdxToNnPointIdx[i, 1]
        
        currIndex = nFeatsAltRef

        currDist = centerIdxToNnDist[i]

        value = distToScore(currDist, pointIdx, dirSignBox, maxDist)
        nrAdded = addTriple(counter, returnIdxs, returnValues, centerIdx, currIndex, value)
        counter += nrAdded
        currIndex += 1

        resId = boxResIds[ pointIdx ]

        currEvoArray = protEvoArray[ resId, : ]

        nrAdded = addTriple_batch(counter, returnIdxs, returnValues, centerIdx, currIndex, currEvoArray)
        counter += nrAdded
        currIndex += nfeatsEvo
        
        currProtQualArray = protQualArray[ resId, : ]

        nrAdded = addTriple_batch(counter, returnIdxs, returnValues, centerIdx, currIndex, currProtQualArray)
        counter += nrAdded
        currIndex += nfeatsQual

        nnPointIdxs = centerIdxToNnPointIdxPerAA[i, :, :]

        for j in range(nnPointIdxs.shape[0]):
            for k in range(nnPointIdxs.shape[1]):
                currNnPointIdx = nnPointIdxs[j, k]
                
                if currNnPointIdx != -1:
                    
                    aaIdx = j
                    atomTypeIndex = k
                    
                    dist = centerIdxToNnDistPerAA[i, aaIdx, atomTypeIndex]

                    value = distToScore(dist, currNnPointIdx, dirSignBox, maxDist)

                    nrAdded = addTriple(counter, returnIdxs, returnValues, centerIdx, (atomTypeIndex*aaIdx)+currIndex, value)
                    counter += nrAdded

    return returnIdxs[0:counter,:], returnValues[0:counter]

# import numba as nb
# from numba import prange

def voxelizeFromTriples(tripleIdx, 
                        tripleVals, 
                        tripleLengthsCumsum, 
                        tripleIdxGlobal,
                        tripleValsGlobal,
                        tripleLengthsGlobalCumsum,
                        nFeats, 
                        nVoxels_local):

    """
    triple로 표현된 sparse feature들을 5D tensor로 변환.
    (배치, x, y, z, feature) 형태

    Returns:
        5D voxel representation: (N, X, Y, Z, F)
    """
    
    nVars = tripleLengthsCumsum.shape[0] - 1

    assert( tripleLengthsCumsum.shape[0] == tripleLengthsGlobalCumsum.shape[0])
    
    nVoxels_local = int(nVoxels_local)


    occs = np.zeros( (nVars, nVoxels_local*nVoxels_local*nVoxels_local, nFeats), dtype=np.float32 )

    # for i in prange(nVars):
    for i in range(nVars):
        
        tripleIdxStart = tripleLengthsCumsum[i]
        tripleIdxEnd = tripleLengthsCumsum[i+1]
        
        for j in range(tripleIdxStart, tripleIdxEnd):
            
            currTripleIdx = tripleIdx[j]
            
            currCenterIdx = currTripleIdx[0]
            currFeatIdx = currTripleIdx[1]
            currTripleVal = tripleVals[j]

            occs[i, currCenterIdx, currFeatIdx] = currTripleVal
            

        tripleIdxGlobalStart = tripleLengthsGlobalCumsum[i]
        tripleIdxGlobalEnd = tripleLengthsGlobalCumsum[i+1]
        
        for j in range(tripleIdxGlobalStart, tripleIdxGlobalEnd):
            
            currFeatIdx = tripleIdxGlobal[j]
            currTripleVal = tripleValsGlobal[j]

            occs[i, :, currFeatIdx] = currTripleVal

    occs_reshaped = occs.reshape( nVars, nVoxels_local, nVoxels_local, nVoxels_local, nFeats )
    
    return occs_reshaped

def getOccTuplesWithAssert_jigsaw( centerIdxToNnPointIdx, 
            centerIdxToNnDist, 
            centerIdxToNnDistPerAA, 
            centerIdxToNnPointIdxPerAA,
            dirSignBox, 
            protEvoArray, 
            protQualArray, 
            boxResIds, 
            nfeatsEvo, 
            nfeatsQual, 
            nfeatsSeq,
            nTargetAtoms,
            maxDist, 
            nFeatsAltRef,
            includeEvoProf,
            includeAaDists):
            #caCbMap):

    resIds = np.zeros((10000), dtype=np.int32)

    returnIdxs = np.full((centerIdxToNnPointIdx.shape[0] * (1 + nfeatsEvo + nfeatsQual + nfeatsSeq*nTargetAtoms), 2), -1, dtype=np.uint16)
    returnValues = np.full((centerIdxToNnPointIdx.shape[0] * (1 + nfeatsEvo + nfeatsQual + nfeatsSeq*nTargetAtoms)), -1, dtype=np.float32)

    counter = 0

    for i in range(centerIdxToNnPointIdx.shape[0]):
        
        centerIdx = centerIdxToNnPointIdx[i, 0]
        pointIdx = centerIdxToNnPointIdx[i, 1]

        currIndex = nFeatsAltRef

        currDist = centerIdxToNnDist[i]

        assert(centerIdx >= 0)

        value = distToScore(currDist, pointIdx, dirSignBox, maxDist)

        nrAdded = addTriple(counter, returnIdxs, returnValues, centerIdx, currIndex, value)

        counter += nrAdded
        currIndex += 1


        resId = boxResIds[ pointIdx ]

        resIds[resId] = 1

        if includeEvoProf:
            currEvoArray = protEvoArray[ resId, : ]

            nrAdded = addTriple_batch(counter, returnIdxs, returnValues, centerIdx, currIndex, currEvoArray)

            counter += nrAdded
            currIndex += nfeatsEvo

        currProtQualArray = protQualArray[ resId, : ]

        nrAdded = addTriple_batch(counter, returnIdxs, returnValues, centerIdx, currIndex, currProtQualArray)

        counter += nrAdded
        currIndex += nfeatsQual


        if includeAaDists:
            nnPointIdxs = centerIdxToNnPointIdxPerAA[i, :, :]
            for j in range(nnPointIdxs.shape[0]):
                for k in range(nnPointIdxs.shape[1]):
                    currNnPointIdx = nnPointIdxs[j, k]

                    if currNnPointIdx != -1:
                        
                        aaIdx = j
                        atomTypeIndex = k
                        
                        dist = centerIdxToNnDistPerAA[i, aaIdx, atomTypeIndex]

                        value = distToScore(dist, currNnPointIdx, dirSignBox, maxDist)

                        nrAdded = addTriple(counter, returnIdxs, returnValues, centerIdx, (nnPointIdxs.shape[1] * aaIdx) + atomTypeIndex + currIndex, value)

                        counter += nrAdded


    assert(np.all(returnValues[0:counter] <= 50.0))
    assert(np.all(returnValues[0:counter] >= -65.0))

    resCount = np.sum(resIds)

    return resCount, returnIdxs[0:counter,:], returnValues[0:counter]


def calcNN_jigsaw(pointNeighborIdxs, centerIdxOrder, dists, boxAtomNamesNum, boxAAs, nTargetAtoms, includePerAaDist):



    prevCenterIdx = -1
    centerCounter = 0
    for i in range(centerIdxOrder.shape[0]):
        currRowIdx = centerIdxOrder[i]
        
        currCenterIdx = pointNeighborIdxs[currRowIdx, 0]
        
        if currCenterIdx == -1: 
            continue
            
        if currCenterIdx != prevCenterIdx and prevCenterIdx != -1:
            centerCounter += 1
    
        prevCenterIdx = currCenterIdx

    if prevCenterIdx != -1:
        centerCounter += 1

    

    centerIdxToNnPointIdx = np.full((centerCounter, 2), -1, dtype=np.int32)
    centerIdxToNnDist = np.full((centerCounter), -1.0, dtype=np.float32)

    centerIdxToNnPointIdxPerAA = np.full((centerCounter, 21, nTargetAtoms), -1, dtype=np.int32)
    centerIdxToNnDistPerAA = np.full((centerCounter, 21, nTargetAtoms), 10000.0, dtype=np.float32)
    
    bestDists = np.full((21), 10000, dtype=np.float32)
    bestPointIdxs = np.full((21), -1, dtype=np.float32)
    

    prevCenterIdx = -1
    bestDist = 10000
    bestPointIdx = -1
    centerCounter = 0
    for i in range(centerIdxOrder.shape[0]):
        currRowIdx = centerIdxOrder[i]
        
        currCenterIdx = pointNeighborIdxs[currRowIdx, 0]
        
        if currCenterIdx == -1: 
            continue
        
        if prevCenterIdx != currCenterIdx and prevCenterIdx != -1:
            centerIdxToNnPointIdx[centerCounter, 0] = prevCenterIdx
            centerIdxToNnPointIdx[centerCounter, 1] = bestPointIdx
            centerIdxToNnDist[centerCounter] = bestDist
            
            bestDist = 10000.0
            bestPointIdx = -1
            bestDists[:] = 10000.0
            bestPointIdxs[:] = -1

            centerCounter += 1
            

        currPointIdx = pointNeighborIdxs[currRowIdx, 1]
        currDist = dists[currRowIdx]
    
        if currDist < bestDist:
            bestDist = currDist
            bestPointIdx = currPointIdx

        if includePerAaDist:
            currAaIdx = boxAAs[currPointIdx]
            currAtomTypeIdx = boxAtomNamesNum[currPointIdx]

            if currAtomTypeIdx != -1:

                oldDist = centerIdxToNnDistPerAA[centerCounter, currAaIdx, currAtomTypeIdx]
                    
                if currDist < oldDist:
                    centerIdxToNnDistPerAA[centerCounter, currAaIdx, currAtomTypeIdx] = currDist

                    centerIdxToNnPointIdxPerAA[centerCounter, currAaIdx, currAtomTypeIdx] =  currPointIdx

        prevCenterIdx = currCenterIdx
    
    if prevCenterIdx != -1:
        centerIdxToNnPointIdx[centerCounter, 0] = prevCenterIdx
        centerIdxToNnPointIdx[centerCounter, 1] = bestPointIdx
        centerIdxToNnDist[centerCounter] = bestDist


    return centerIdxToNnPointIdx, centerIdxToNnDist, centerIdxToNnDistPerAA, centerIdxToNnPointIdxPerAA


def calcNN_jigsaw(pointNeighborIdxs, centerIdxOrder, dists, boxAtomNamesNum, boxAAs, nTargetAtoms, includePerAaDist):



    prevCenterIdx = -1
    centerCounter = 0
    for i in range(centerIdxOrder.shape[0]):
        currRowIdx = centerIdxOrder[i]
        
        currCenterIdx = pointNeighborIdxs[currRowIdx, 0]
        
        if currCenterIdx == -1: 
            continue
            
        if currCenterIdx != prevCenterIdx and prevCenterIdx != -1:
            centerCounter += 1
    
        prevCenterIdx = currCenterIdx

    if prevCenterIdx != -1:
        centerCounter += 1

    

    centerIdxToNnPointIdx = np.full((centerCounter, 2), -1, dtype=np.int32)
    centerIdxToNnDist = np.full((centerCounter), -1.0, dtype=np.float32)

    centerIdxToNnPointIdxPerAA = np.full((centerCounter, 21, nTargetAtoms), -1, dtype=np.int32)
    centerIdxToNnDistPerAA = np.full((centerCounter, 21, nTargetAtoms), 10000.0, dtype=np.float32)
    
    bestDists = np.full((21), 10000, dtype=np.float32)
    bestPointIdxs = np.full((21), -1, dtype=np.float32)
    

    prevCenterIdx = -1
    bestDist = 10000
    bestPointIdx = -1
    centerCounter = 0
    for i in range(centerIdxOrder.shape[0]):
        currRowIdx = centerIdxOrder[i]
        
        currCenterIdx = pointNeighborIdxs[currRowIdx, 0]
        
        if currCenterIdx == -1: 
            continue
        
        if prevCenterIdx != currCenterIdx and prevCenterIdx != -1:
            centerIdxToNnPointIdx[centerCounter, 0] = prevCenterIdx
            centerIdxToNnPointIdx[centerCounter, 1] = bestPointIdx
            centerIdxToNnDist[centerCounter] = bestDist
            
            bestDist = 10000.0
            bestPointIdx = -1
            bestDists[:] = 10000.0
            bestPointIdxs[:] = -1

            centerCounter += 1
            

        currPointIdx = pointNeighborIdxs[currRowIdx, 1]
        currDist = dists[currRowIdx]
    
        if currDist < bestDist:
            bestDist = currDist
            bestPointIdx = currPointIdx

        if includePerAaDist:
            currAaIdx = boxAAs[currPointIdx]
            currAtomTypeIdx = boxAtomNamesNum[currPointIdx]

            if currAtomTypeIdx != -1:

                oldDist = centerIdxToNnDistPerAA[centerCounter, currAaIdx, currAtomTypeIdx]
                    
                if currDist < oldDist:
                    centerIdxToNnDistPerAA[centerCounter, currAaIdx, currAtomTypeIdx] = currDist

                    centerIdxToNnPointIdxPerAA[centerCounter, currAaIdx, currAtomTypeIdx] =  currPointIdx

        prevCenterIdx = currCenterIdx
    
    if prevCenterIdx != -1:
        centerIdxToNnPointIdx[centerCounter, 0] = prevCenterIdx
        centerIdxToNnPointIdx[centerCounter, 1] = bestPointIdx
        centerIdxToNnDist[centerCounter] = bestDist


    return centerIdxToNnPointIdx, centerIdxToNnDist, centerIdxToNnDistPerAA, centerIdxToNnPointIdxPerAA

def dist3D(coords_x, coords_y):

    val = 0

    tmp = coords_x[0] - coords_y[0]
    val += tmp * tmp

    tmp = coords_x[1] - coords_y[1]
    val += tmp * tmp

    tmp = coords_x[2] - coords_y[2]
    val += tmp * tmp

    return val


def calcDists(pointNeighborIdxs, coords, centers):
    maxIdx = 0
    for i in range(pointNeighborIdxs.shape[0]):
        if pointNeighborIdxs[i, 0] == -1:
            break
        maxIdx += 1

    dists = np.full((pointNeighborIdxs.shape[0]), -1, dtype=np.float32)

    for i in range(maxIdx):

        centerIdx = pointNeighborIdxs[i, 0]
        pointIdx = pointNeighborIdxs[i, 1]

        pointCoords = coords[ pointIdx ]
        centerCoords = centers[ centerIdx ]

        disti = dist3D(pointCoords, centerCoords)

        dists[i] = disti

    return dists

def getNearestBoxCenter(coord, boxLen_half, voxelSize_local, maxVoxelIdx):
    
    idxs = np.empty(3, dtype=np.int32)
    
    coord_std = coord[0] + boxLen_half
    idxs[0] = max(0, min(maxVoxelIdx, int(coord_std / voxelSize_local)))
    
    coord_std = coord[1] + boxLen_half
    idxs[1] = max(0, min(maxVoxelIdx, int(coord_std / voxelSize_local)))

    coord_std = coord[2] + boxLen_half
    idxs[2] = max(0, min(maxVoxelIdx, int(coord_std / voxelSize_local)))
    
    return idxs

def getPointNeighborIdxs(coords, centerIdxToNeighborIdxs, boxLen_half, voxelSize_local, maxVoxelIdx, nVoxels_local):

    pointNeighborIdxs = np.full((coords.shape[0]*3*3*3, 2), -1, dtype=np.int32)

    pointToCenter = np.empty((coords.shape[0], 2), dtype=np.int32)
    for coord_i in range(coords.shape[0]):
        boxCenter = getNearestBoxCenter(coords[coord_i,:], boxLen_half, voxelSize_local, maxVoxelIdx)
        boxCenterIdx = getCenterIdx(boxCenter, nVoxels_local)
        pointToCenter[coord_i, 0] = coord_i
        pointToCenter[coord_i, 1] = boxCenterIdx

    counter = 0
    for i in range(pointToCenter.shape[0]):
        
        boxCenterIdx = pointToCenter[i, 1]
        pointIdx = pointToCenter[i, 0]
               
        neighborCenterIdxs = centerIdxToNeighborIdxs[boxCenterIdx, :]

        pointNeighborIdxs[ counter, 0 ] = boxCenterIdx
        pointNeighborIdxs[ counter, 1 ] = pointIdx

        counter += 1

        for neighbor_i in range(neighborCenterIdxs.shape[0]):
            if neighborCenterIdxs[ neighbor_i ] != -1:
                pointNeighborIdxs[ counter, 0 ] = neighborCenterIdxs[ neighbor_i ]
                pointNeighborIdxs[ counter, 1 ] = pointIdx

                #evoProfs[counter, :] = evoProf
                counter += 1


    return pointNeighborIdxs


def getNFeats(nFeatsSeq,
            nTargetAtoms,
            nFeatsEvo,
            nFeatsAlt,
            nFeatsAllAtomDist,
            nFeatsProtQual,
            includeEvoProfs,
            includeAlt,
            includeAllAtomDist,
            includeProtQual):

    nFeats = nFeatsSeq * nTargetAtoms
    
    if includeEvoProfs:
        nFeats += nFeatsEvo
    if includeAlt:
        nFeats += nFeatsAlt
    if includeProtQual:
        nFeats += nFeatsProtQual
    if includeAllAtomDist:
        nFeats += nFeatsAllAtomDist
    
    return nFeats

In [2]:
import pickle

SAMPLE_JIGSAW = 1
SAMPLE_DS = 2

def init():
	global globalVars
	globalVars = {}

def calcOccupancy_triples(  coords, 
                    centerIdxToNeighborIdxs, 
                    boxLen_half, 
                    voxelSize_local, 
                    maxVoxelIdx, 
                    nVoxels_local, 
                    centers, 
                    protEvoArray, 
                    protQualArray, 
                    boxResIds, 
                    dirSignBox, 
                    nfeatsEvo, 
                    nfeatsQual, 
                    nfeatsSeq,
                    nTargetAtoms,
                    maxDist, 
                    nFeatsAltRef, 
                    boxAtomNamesNum,
                    boxAAs,
                    voxelizeWithAsserts,
                    includeEvoProfs,
                    includePerAaDists):
    pointNeighborIdxs = getPointNeighborIdxs(coords, centerIdxToNeighborIdxs, boxLen_half, voxelSize_local, maxVoxelIdx, nVoxels_local)

    dists = calcDists(pointNeighborIdxs, coords, centers)
    
    centersSortedIdxs = np.argsort(pointNeighborIdxs[:,0]).astype(np.int32)
    
    centerIdxToNnPointIdx, centerIdxToNnDist, centerIdxToNnDistPerAA, centerIdxToNnPointIdxPerAA = calcNN_jigsaw(pointNeighborIdxs, centersSortedIdxs, dists, boxAtomNamesNum, boxAAs, nTargetAtoms, includePerAaDists)

    if voxelizeWithAsserts:
        centerIdxToResIds = np.empty(centerIdxToNnPointIdx.shape, dtype=np.uint16)
        centerIdxToResIds[:,0] = centerIdxToNnPointIdx[:, 0]
        centerIdxToResIds[:,1] = boxResIds[centerIdxToNnPointIdx[:, 1]]

        resIdCount, tripleIdxs, tripleVals =  getOccTuplesWithAssert_jigsaw(    centerIdxToNnPointIdx, 
                                                                                centerIdxToNnDist, 
                                                                                centerIdxToNnDistPerAA, 
                                                                                centerIdxToNnPointIdxPerAA,
                                                                                dirSignBox, 
                                                                                protEvoArray, 
                                                                                protQualArray, 
                                                                                boxResIds, 
                                                                                nfeatsEvo, 
                                                                                nfeatsQual,
                                                                                nfeatsSeq,
                                                                                nTargetAtoms,
                                                                                maxDist, 
                                                                                nFeatsAltRef,
                                                                                includeEvoProfs,
                                                                                includePerAaDists)



    else:
        raise Exception("Not implemented")

    return resIdCount, tripleIdxs, tripleVals, centerIdxToResIds
    


def voxelize_triples(   pdbTxn,
                targetSnpDF_local_oneRow, 
                centers, 
                edgeLen, 
                c,
                centerIdxToNeighborCoords,
                centerIdxToNeighborIdxs,
                maxVoxelIdx,
                boxLen_half,
                voxelSize_local,
                nVoxels_local):

    gene_name = targetSnpDF_local_oneRow[0]
    change_position_1based = targetSnpDF_local_oneRow[1]

    isJigsaw = targetSnpDF_local_oneRow[4] == SAMPLE_JIGSAW

    targetPdbObjectBytes = pdbTxn.get(gene_name.encode("ascii"))
    targetPdbObject = pickle.loads(targetPdbObjectBytes)

    centralCaAtomCoords = targetPdbObject.get('caArray')[change_position_1based]
    centralCaAtomIdx = targetPdbObject.get('caIndexArray')[change_position_1based][0]

    try:
        assert(targetPdbObject["resid"][int(centralCaAtomIdx)] == change_position_1based)
    except AssertionError:
        real_pos = targetPdbObject["resid"][int(centralCaAtomIdx)]
        print(f"❌ Assertion failed: resid={real_pos} vs MutPos={change_position_1based}")
        raise

    dirSignArray = getDirSignArray(targetPdbObject.get('element').shape[0], centralCaAtomIdx)

    allCoords = targetPdbObject.get('coords')
    allCoords = centerAndRandomRotate(allCoords, centralCaAtomCoords, globalVars["rotMatrices"], c["rotate"])

    if isJigsaw:
        boxIdx = getTargetBoxIdxs( allCoords, targetPdbObject["resid"], change_position_1based, edgeLen, c["excludeCentralAA"] )
    else:
        boxIdx = getTargetBoxIdxs( allCoords, targetPdbObject["resid"], change_position_1based, edgeLen, False )

    boxCoords = allCoords[boxIdx]
    boxResIds = targetPdbObject["resid"][boxIdx].astype(np.int32)
    boxAtomNames = targetPdbObject['name'][boxIdx]
    boxAAs = targetPdbObject["resnamenum"][boxIdx].astype(np.int32)
    protEvoArray = np.ascontiguousarray(targetPdbObject["feat_cons"])
    protQualArray = np.ascontiguousarray(targetPdbObject["qualArray"])

    if "doPai" in c and ((not c["doPai"]) or (not c["doJigsaw"])):
        protEvoArray[:,21] = 0
        protEvoArray[:,22] = 1
    elif isJigsaw:
        protEvoArray[:,21] = 1
        protEvoArray[:,22] = 0
    else:
        protEvoArray[:,21] = 0
        protEvoArray[:,22] = 1

    if not c["excludeCentralAA"] and isJigsaw:
        raise Exception("not implemented")

    dirSignsBox = dirSignArray[boxIdx]

    targetAtoms = c["targetAtoms"]
    boxAtomNamesNum =  np.ascontiguousarray(np.full(boxAtomNames.shape[0], -1,  dtype=np.int32))
    for i, targetAtom in enumerate(targetAtoms):
        idxs = np.where(boxAtomNames == targetAtom)
        boxAtomNamesNum[idxs] = i


    resIdCount, tripleIdxs, tripleVals, centerIdxToResIds = calcOccupancy_triples(boxCoords,
                    centerIdxToNeighborIdxs, 
                    boxLen_half, 
                    voxelSize_local, 
                    maxVoxelIdx,
                    nVoxels_local, 
                    centers, 
                    protEvoArray, 
                    protQualArray, 
                    boxResIds, 
                    dirSignsBox, 
                    c["nFeatsEvo"], 
                    c["nFeatsProtQual"],
                    c["nFeatsSeq"],
                    len(c["targetAtoms"]),
                    c["distanceUpperBound"] * c["distanceUpperBound"], 
                    c["nFeatsAltRef"], 
                    boxAtomNamesNum,
                    boxAAs,
                    c["voxelizeWithAsserts"],
                    c["includeEvoProfs"],
                    c["includePerAaDists"])

    idxsGlobals = np.zeros((2), dtype=np.uint16)
    valsGlobal = np.zeros((2), dtype=np.float32)


    countsTuple = None
    if c["voxelizeWithAsserts"]:
        countsTuple = (np.unique(boxResIds).shape[0], resIdCount)

    return tripleIdxs.astype(np.uint16), tripleVals, idxsGlobals, valsGlobal, countsTuple, centerIdxToResIds

def getSampleWeight(posTuple, pdbRepoDict, c):
    
    protId, protPos, protPosResnamenum, labelsArr, sampleType = posTuple

    if sampleType == SAMPLE_JIGSAW:
        return c["jigsawSampleWeight"]
    elif sampleType == SAMPLE_DS:
        return 1.0
    else:
        return 1.0

def getLabelArrForPos(posTuple, pdbRepoDict, c):
    protId, protPos, protPosResnamenum, labelsArr, isJigsaw = posTuple


    return labelsArr

def getVoxelGridNN(centerIdxToResIds, nVoxels_local):
    voxelGrid_nn = np.full((len(centerIdxToResIds), nVoxels_local * nVoxels_local * nVoxels_local, 1), -1, dtype=np.int32)

    for i in range(len(centerIdxToResIds)):
        centerIdxToResId = centerIdxToResIds[i]

        voxelGrid_nn[i, centerIdxToResId[:, 0], 0] = centerIdxToResId[:, 1]

    voxelGrid_nn = voxelGrid_nn.reshape((len(centerIdxToResIds), nVoxels_local, nVoxels_local, nVoxels_local, 1))

    return voxelGrid_nn

def getVoxelGridToNNMap(centerIdxToResIds, nvoxels, geneNameToId):

    geneIDs = []
    centerIdxToResIdLocals = []

    for gene_name, centerIdxToResIdLocal in centerIdxToResIds:

        centerIdxToResIdLocals.append(centerIdxToResIdLocal)
        geneIDs.append( geneNameToId[gene_name] )


    geneIdsArr = np.array(geneIDs)
    voxelGridNNs = getVoxelGridNN(centerIdxToResIdLocals, nvoxels)

    return geneIdsArr, voxelGridNNs

def concatenateLabels(labels):
    firstLabel = labels[0]
    
#     print("First")
#     print(firstLabel)
#     print(firstLabel.shape)
#     print(len(firstLabel.shape))
    
    if firstLabel.shape[0] > 1 and len(firstLabel.shape) > 1:
        newLabels = []
        for dimi in range(firstLabel.shape[0]):
            currLabels = []
            for labeli in labels:
                currLabels.append( labeli[dimi][np.newaxis,:] )
            newLabels.append( np.concatenate(currLabels) )
                        
        return newLabels
    else:
        newLabels = []
        for labeli in labels:
            newLabels.append(labeli[np.newaxis, :])
    
        return [np.concatenate(newLabels)]

def data_generation_triples(snpRowList_temp, config, pdbLmdb, epoch=0, counts=False):
    'Generates data containing batch_size samples' # X : (n_samples, *dim, n_channels)

    X = []
    countTuples = []

    nVoxels = np.array(config["nVoxels"])
    boxSize = (nVoxels * config["voxelSize"]).astype("float32")

    edgeLen = ((boxSize[0] / 2) + 4.6)
    centers = getGridCenters(nVoxels, np.array([0, 0, 0]), config["voxelSize"]).astype("float32")

    boxLen = config["nVoxels"][0] * config["voxelSize"]

    voxelSize_local = np.float32(config["voxelSize"])
    boxLen_half = np.float32(boxLen / 2)
    nVoxels_local = int(config["nVoxels"][0])
    maxVoxelIdx = config["nVoxels"][0] - 1

    centerIdxToNeighborCoords, centerIdxToNeighborIdxs, allGood = getCenterNeighbors(centers, voxelSize_local, boxLen_half, nVoxels_local)

    if not allGood:
        raise Exception("This should not happen")

    tripleIdxss = []
    tripleValss = []
    tripleLengths = []
    tripleIdxGlobals = []
    tripleValsGlobals = []
    tripleLengthsGlobals = []

    centerIdxToResIds_list = []


    with pdbLmdb.begin() as pdbTxn:
        for i in range(snpRowList_temp.shape[0]):

            tripleIdxs, tripleVals, tripleIdxGlobal, tripleValsGlobal, countsTuple, centerIdxToResIds = voxelize_triples( pdbTxn,
                                                                    snpRowList_temp[i],
                                                                    centers,
                                                                    edgeLen,
                                                                    config,
                                                                    centerIdxToNeighborCoords,
                                                                    centerIdxToNeighborIdxs,
                                                                    maxVoxelIdx,
                                                                    boxLen_half,
                                                                    voxelSize_local,
                                                                    nVoxels_local)


            tripleIdxss.append(tripleIdxs)
            tripleValss.append(tripleVals)
            tripleLengths.append(tripleIdxs.shape[0])

            tripleIdxGlobals.append(tripleIdxGlobal)
            tripleValsGlobals.append(tripleValsGlobal)
            tripleLengthsGlobals.append(tripleIdxGlobal.shape[0])

            countTuples.append( tuple(list(snpRowList_temp[i]) + list(countsTuple) ) )

            centerIdxToResIds_list.append( (snpRowList_temp[i][0], centerIdxToResIds) )


    tripleIdxAll = np.concatenate(tripleIdxss)
    tripleValsAll = np.concatenate(tripleValss)
    tripleLengthsAll = np.array(tripleLengths)
    tripleIdxGlobalAll = np.concatenate(tripleIdxGlobals)
    tripleValsGlobalAll = np.concatenate(tripleValsGlobals)
    tripleLengthsGlobalAll = np.array(tripleLengthsGlobals)

    #tripleLengthsAll = np.concatenate([np.array([0]), np.cumsum(tripleLengthsAll)])  #tripleLengths  #tripleLengths
    #tripleLengthsGlobalAll = np.concatenate([np.array([0]), np.cumsum(tripleLengthsGlobalAll)])  #tripleLengthsGlobal
    
    X = [tripleIdxAll, 
         tripleValsAll, 
         tripleLengthsAll, 
         tripleIdxGlobalAll,
         tripleValsGlobalAll,
         tripleLengthsGlobalAll]

    y_list = []
    sample_weights = []
    for i in range(snpRowList_temp.shape[0]):
        y_list.append( getLabelArrForPos(snpRowList_temp[i], pdbLmdb, config)  )
        sample_weights.append( getSampleWeight( snpRowList_temp[i], pdbLmdb, config ) )

    y = concatenateLabels(y_list)

    multiz_geneNames, multiz_voxelGridNNs = getVoxelGridToNNMap(centerIdxToResIds_list, config["nVoxels"][0], globalVars.globalVars["geneNameToId"])
    #multiz_varArr, multiz_voxelGridNNs = getMultizData(centerIdxToResIds_list, multizLmdb, config["nVoxels"][0] )

    returnTuple = ( X if len(X) > 1 else X[0], 
                    y if len(y) > 1 else y[0],
                    countTuples,
                    sample_weights,
                    multiz_geneNames,
                    multiz_voxelGridNNs)

    returnTuple_compressed = returnTuple #zlib.compress(pickle.dumps( returnTuple ))

    return returnTuple_compressed

In [3]:
import pandas as pd
import numpy as np
import os
import time

class AllEval():
    def __init__(self, c, variantsDfFilePath, pdbLmdb_path, verbose=0):
        super(AllEval, self).__init__()

        pdbLmdb = lmdb.open(pdbLmdb_path, create=False, subdir=True, readonly=True, lock=False)

        allDF = pd.read_csv(variantsDfFilePath, index_col=0)

        if not "name" in allDF.columns:
            allDF["name"] = allDF["gene_name"]

        allDF = allDF[~allDF["name"].isna()].copy()

        allRows_tmpDF = allDF[["name", "change_position_1based"]].drop_duplicates()# #

        allRows = []
        allRows_jigsaw = []
        for name, change_position_1based in allRows_tmpDF.values.tolist():
            labelArr = np.zeros(20, dtype=np.float32)
            label_numeric_aa = 0

            isJigsaw = False
            allRows.append( (name, change_position_1based, label_numeric_aa, labelArr, isJigsaw) )
            isJigsaw = True
            allRows_jigsaw.append( (name, change_position_1based, label_numeric_aa, labelArr, isJigsaw) )

        print("Voxelizing PAI")
        X_eval = self.voxelizeAll(allRows, c, pdbLmdb)
        print("Voxelizing Jigsaw")
        X_eval_jigsaw = self.voxelizeAll(allRows_jigsaw, c, pdbLmdb)

        self.evalName = os.path.basename(variantsDfFilePath).replace(".csv", "")
        
        self.X_eval = X_eval
        self.X_eval_jigsaw = X_eval_jigsaw
        self.c = c
        self.epoch = []
        self.history = {}
        self.verbose = verbose
        self.allDF = allDF
        self.allRows_tmpDF = allRows_tmpDF

        pdbRepoDict=None


    def voxelizeAll(self, rows, c, pdbLmdb):
        X_chunk, y_chunk, countTuples, sampleWeights, multiz_geneNames, multiz_voxelGridNNs = data_generation_triples(np.array(rows, dtype="object"), c, pdbLmdb, epoch=0, counts=False)

        nFeats = getNFeats( c["nFeatsSeq"],
                            len(c["targetAtoms"]),
                            c["nFeatsEvo"],
                            c["nFeatsAltRef"],
                            c["nFeatsAllAtomDist"],
                            c["nFeatsProtQual"],                    
                            c["includeEvoProfs"],
                            c["includeAlt"],
                            c["includeAllAtomDist"],
                            c["includeProtQual"])

        X_chunk[2] = np.concatenate([np.array([0]), np.cumsum(X_chunk[2])])  #tripleLengths
        X_chunk[5] = np.concatenate([np.array([0]), np.cumsum(X_chunk[5])])  #tripleLengthsGlobal

        print("Restoring voxels")
        X = voxelizeFromTriples(X_chunk[0], 
                                X_chunk[1], 
                                X_chunk[2], 
                                X_chunk[3],
                                X_chunk[4],
                                X_chunk[5],
                                nFeats, 
                                np.float32(c["nVoxels"][0])).astype(np.float32)

        print("Val data shape voxels: %s" % ( str(X.shape) ) )  #str(X_val[1].shape),  %s
        if c["doMultiz"]:
            X = [X,
                 multiz_geneNames,
                 multiz_voxelGridNNs]

        return X


    def performEval(self, model, savePredFile=None):

        t = time.time()

        print("Starting all eval")

        y_pred_eval = model.predict(self.X_eval)
        y_pred_eval_jigsaw = model.predict(self.X_eval_jigsaw)
        y_pred_eval_both = np.mean([y_pred_eval, y_pred_eval_jigsaw], axis=0)

        self.allRows_tmpDF.reset_index(inplace=True, drop=True)
        self.allRows_tmpDF.reset_index(inplace=True, drop=False)

        allDF_merged = self.allDF.merge(self.allRows_tmpDF, on=["name", "change_position_1based"])

        alt_score = y_pred_eval[ allDF_merged["index"].values, allDF_merged.label_numeric_aa_alt.values ]
        alt_score_jigsaw = y_pred_eval_jigsaw[ allDF_merged["index"].values, allDF_merged.label_numeric_aa_alt.values ]
        alt_score_both = y_pred_eval_both[ allDF_merged["index"].values, allDF_merged.label_numeric_aa_alt.values ]

        fullScoreDF = pd.DataFrame({"snp_id": allDF_merged.snp_id, "predPai": alt_score, "predJigsaw": alt_score_jigsaw, "predBoth": alt_score_both})

        print("Done all eval (%s)" % str( time.time() - t ))
        
        return fullScoreDF

In [4]:
from Bio.PDB import PDBParser
import numpy as np
import os

def parse_alphafold_pdb_to_pdbobject(pdb_path):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("af", pdb_path)

    atom_coords = []
    atom_names = []
    res_ids = []
    res_names = []
    ca_coords = []
    ca_indices = []
    res_id_set = set()
    
    for model in structure:
        for chain in model:
            for res in chain:
                hetfield, resseq, icode = res.get_id()
                if hetfield != " ":  # skip HETATM
                    continue
                res_id = resseq
                resname = res.get_resname()
                one_letter = aa3_to_num.get(resname, 20)  # unknown=20

                for atom in res:
                    atom_coords.append(atom.coord)
                    atom_names.append(atom.get_name())
                    res_ids.append(res_id)
                    res_names.append(one_letter)

                    if atom.get_name() == "CA":
                        ca_coords.append(atom.coord)
                        ca_indices.append(len(atom_coords)-1)

    pdbObject = {
        'coords': np.array(atom_coords, dtype=np.float32),
        'name': np.array(atom_names),
        'resid': np.array(res_ids),
        'resnamenum': np.array(res_names),
        'caArray': np.array(ca_coords, dtype=np.float32),
        'caIndexArray': np.array(ca_indices).reshape(-1, 1),
        'feat_cons': np.zeros((len(ca_coords), 23), dtype=np.float32),   # placeholder
        'qualArray': np.ones((len(ca_coords), 10), dtype=np.float32) * 0.9,  # placeholder
    }
    return pdbObject

# 3-letter AA → integer
aa3_to_num = {
    'ALA': 0, 'CYS': 1, 'ASP': 2, 'GLU': 3, 'PHE': 4,
    'GLY': 5, 'HIS': 6, 'ILE': 7, 'LYS': 8, 'LEU': 9,
    'MET': 10, 'ASN': 11, 'PRO': 12, 'GLN': 13, 'ARG': 14,
    'SER': 15, 'THR': 16, 'VAL': 17, 'TRP': 18, 'TYR': 19
}

class PDBFileLoader:
    def __init__(self, pdb_dir):
        self.pdb_cache = {}
        self.pdb_dir = pdb_dir

    def get(self, gene_bytes):
        gene_name = gene_bytes.decode("ascii")
        if gene_name in self.pdb_cache:
            return self.pdb_cache[gene_name]

        matches = [f for f in os.listdir(self.pdb_dir) if f.startswith("AF-" + gene_name)]
        if not matches:
            raise FileNotFoundError(f"No PDB file found for {gene_name}")
        
        pdb_path = os.path.join(self.pdb_dir, matches[0])
        pdb_obj = parse_alphafold_pdb_to_pdbobject(pdb_path)
        raw = pickle.dumps(pdb_obj)
        self.pdb_cache[gene_name] = raw
        return raw


In [5]:
def get_config():
    return {
        # voxel size and grid
        "nVoxels": [7, 7, 7],
        "voxelSize": 1.0,

        # center filtering
        "excludeCentralAA": True,
        "distanceUpperBound": 5.0,

        # atom-level encoding
        "targetAtoms": ["CA", "CB"],

        # feature dimensions
        "nFeatsSeq": 21,          # amino acid one-hot or embedding
        "nFeatsEvo": 46,          # MSA-based features (e.g. feat_prof + feat_cons)
        "nFeatsAltRef": 42,       # not used in voxelize, but still needed for slot indexing
        "nFeatsAllAtomDist": 1,   # distance to each AA-atom type
        "nFeatsProtQual": 10,     # pLDDT etc.

        # feature toggle flags
        "includeEvoProfs": True,
        "includeAlt": True,
        "includeAllAtomDist": True,
        "includeProtQual": True,
        "includePerAaDists": True,

        # voxelization mode
        "voxelizeWithAsserts": True,
        "rotate": True,

        # evaluation (not needed for voxelize, but harmless)
        "doMultiz": False
    }


In [6]:
import pandas as pd

tsv_path = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\rhapsody2_sav_db_exactmatch_only.tsv"
pdb_dir = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\alphafold_structures"

df = pd.read_csv(tsv_path, sep="\t", header=None)
df.columns = ["UniProtID", "StructureFile", "MutPos", "WT", "Mut", "Label"]

pdb_txn = PDBFileLoader(pdb_dir)

In [8]:
import pandas as pd

tsv_path = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\rhapsody2_sav_db_exactmatch_only.tsv"
pdb_dir = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\alphafold_structures"

df = pd.read_csv(tsv_path, sep="\t", header=None)
df.columns = ["UniProtID", "StructureFile", "MutPos", "WT", "Mut", "Label"]

pdb_txn = PDBFileLoader(pdb_dir)
c = get_config()  # 네가 쓰던 config 딕셔너리 반환

# Grid 준비
nVoxels = np.array(c["nVoxels"])
centers = getGridCenters(nVoxels, np.array([0, 0, 0]), c["voxelSize"]).astype("float32")
boxSize = (nVoxels * c["voxelSize"]).astype("float32")
edgeLen = (boxSize[0] / 2) + 4.6
boxLen_half = c["nVoxels"][0] * c["voxelSize"] / 2
voxelSize_local = float(c["voxelSize"])
nVoxels_local = c["nVoxels"][0]
maxVoxelIdx = nVoxels_local - 1
centerIdxToNeighborCoords, centerIdxToNeighborIdxs, _ = getCenterNeighbors(centers, voxelSize_local, boxLen_half, nVoxels_local)

# 한 샘플 예시
row = df.iloc[0]
gene = row["UniProtID"]
mutpos = int(row["MutPos"])
example_tuple = (gene, mutpos, 0, np.zeros(20, dtype=np.float32), False)

tripleIdxs, tripleVals, tripleIdxGlobal, tripleValsGlobal, _, centerIdxToResIds = voxelize_triples(
    pdb_txn,
    example_tuple,
    centers,
    edgeLen,
    c,
    centerIdxToNeighborCoords,
    centerIdxToNeighborIdxs,
    maxVoxelIdx,
    boxLen_half,
    voxelSize_local,
    nVoxels_local
)


voxel_tensor = voxelizeFromTriples(
    tripleIdxs,
    tripleVals,
    np.array([0, len(tripleIdxs)]),
    tripleIdxGlobal,
    tripleValsGlobal,
    np.array([0, len(tripleIdxGlobal)]),
    nFeats=getNFeats(
        c["nFeatsSeq"], len(c["targetAtoms"]), c["nFeatsEvo"],
        c["nFeatsAltRef"], c["nFeatsAllAtomDist"], c["nFeatsProtQual"],
        c["includeEvoProfs"], c["includeAlt"], c["includeAllAtomDist"], c["includeProtQual"]
    ),
    nVoxels_local=nVoxels_local
)

print("Voxel shape:", voxel_tensor.shape)  # (1, 7, 7, 7, C)

❌ Assertion failed: resid=70 vs MutPos=69


AssertionError: 